# Model Development

### Project Overview & Objective
In this notebook, two machine learning models are trained to recognize fraud:

1. Logistic Regression (Baseline): A simple, fast formula that gives a clear score and shows exactly how each clue pushes risk up or down.
2. Random Forest (Ensemble Model): A team of 300 decision trees working together. It spots subtle combinations of clues (such as high velocity from an unfamiliar device during an international wire transfer).

The data is split to test the models fairly, numerical scales are adjusted, both models are trained, and the artifacts are saved for production use.


### 1. Import Machine Learning Libraries
Loads scikit-learn tools to split data, scale numbers, train the models, and save them to files.


In [10]:
import os
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

### 2. Load Engineered Features
Loads data/processed/feature_matrix.csv created in 03_feature_engineering.ipynb.


In [11]:
data_path = os.path.join('..', 'data', 'processed', 'feature_matrix.csv')
df = pd.read_csv(data_path)
print(f'Feature dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
df[['transaction_id', 'log_amount', 'log_amount_to_avg_ratio', 'transaction_velocity_1h', 'is_new_device', 'is_fraud']].head(3)

Feature dataset loaded: 126,000 rows x 50 columns


,transaction_id,log_amount,log_amount_to_avg_ratio,transaction_velocity_1h,is_new_device,is_fraud
0,FLR250216186265,11.341354,2.323368,0,0,0
1,FLR250423143305,9.120889,0.693147,0,0,0
2,FLR250424154897,9.934986,1.181727,0,0,0


### 3. Separate Input Features from the Fraud Label
Separates the target prediction column (whether a payment is fraudulent) from all the clues the model is allowed to use.


In [12]:
exclude_cols = [
    'transaction_id', 'account_id', 'timestamp', 'day_of_week', 'description', 'merchant_name',
    'currency', 'transaction_country', 'home_country', 'device_id', 'status',
    'account_holder_name', 'account_created_date', 'amount', 'abs_amount',
    'personal_spend_baseline_usd', 'is_fraud'
]
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols]
y = df['is_fraud']

print(f'Predictor feature count: {len(feature_cols)}')
print(f'Target distribution: {y.value_counts().to_dict()}')

Predictor feature count: 33
Target distribution: {0: 122648, 1: 3352}


### 4. Split Data into Training and Test Sets
Divides the data into two groups: 75% for the models to study and learn from (the training set, 94,500 records), and 25% kept hidden away to test them fairly afterwards (the holdout test set, 31,500 records). Both groups preserve the exact same 2.66% fraud rate.


In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f'Training set: {len(X_train):,} observations ({y_train.sum():,} fraud cases, {y_train.mean()*100:.2f}%)')
print(f'Holdout test set: {len(X_test):,} observations ({y_test.sum():,} fraud cases, {y_test.mean()*100:.2f}%)')

Training set: 94,500 observations (2,514 fraud cases, 2.66%)
Holdout test set: 31,500 observations (838 fraud cases, 2.66%)


### 5. Scale Numerical Features for Fair Comparison
Some numbers are in the thousands (like dollar amounts), while others are small counts (like 1 to 5 transactions per hour). Rescaling puts all clues on a fair, comparable footing so no single large number dominates just because of its units.


In [14]:
num_cols = ['log_amount', 'log_amount_to_avg_ratio', 'log_avg_30d', 'transaction_velocity_1h', 'account_age_days', 'hour_of_day']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

print('Numerical features standardized (mean=0, std=1).')

Numerical features standardized (mean=0, std=1).


### 6. Train the Baseline Model (Logistic Regression)
Trains the baseline model. Because fraud is rare, the model is configured to pay extra attention when it sees a fraud case so it does not ignore them.


In [15]:
lr_baseline = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
lr_baseline.fit(X_train_scaled, y_train)
print('Baseline Logistic Regression model trained successfully.')

Baseline Logistic Regression model trained successfully.


### 7. Train the Random Forest Model
Trains a Random Forest classifier using 300 decision trees working together, each checking different clues, to produce an accurate combined verdict.


In [16]:
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    class_weight='balanced_subsample',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print('Random Forest Classifier trained successfully.')

Random Forest Classifier trained successfully.


### 8. Save Trained Models and Pipeline Artifacts
Saves the trained models, scaler, and feature list into model/ so the live scoring system and web app can use them anytime.


In [17]:
model_dir = os.path.join('..', 'model')
os.makedirs(model_dir, exist_ok=True)

lr_path = os.path.join(model_dir, 'logistic_regression_baseline.joblib')
rf_path = os.path.join(model_dir, 'random_forest_model.joblib')
scaler_path = os.path.join(model_dir, 'scaler.joblib')
feat_path = os.path.join(model_dir, 'feature_names.joblib')

joblib.dump(lr_baseline, lr_path)
joblib.dump(rf_model, rf_path)
joblib.dump(scaler, scaler_path)
joblib.dump(feature_cols, feat_path)

print(f'Model artifacts saved to {model_dir}:')
print(f' - {os.path.basename(lr_path)}')
print(f' - {os.path.basename(rf_path)}')
print(f' - {os.path.basename(scaler_path)}')
print(f' - {os.path.basename(feat_path)}')

Model artifacts saved to ..\model:
 - logistic_regression_baseline.joblib
 - random_forest_model.joblib
 - scaler.joblib
 - feature_names.joblib


Summary of Model Artifacts:
- logistic_regression_baseline.joblib: Baseline linear model.
- random_forest_model.joblib: Trained Random Forest model saved for downstream evaluation.
- scaler.joblib: Tool used to scale numbers for new payments.
- feature_names.joblib: Complete ordered list of expected features.

---
Next Step: In 05_model_evaluation_and_thresholds.ipynb, both models are tested on unseen data, their accuracy is compared to select the winning champion model, and alert thresholds are set up for fraud analysts.
